In [2]:
import gradio as gr
import json
import os
import uuid
from langchain.llms import Ollama
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

In [3]:
# Thư mục lưu trữ lịch sử chat
DATA_DIR = "chat_history"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Lưu memory riêng cho từng user
user_sessions = {}

# Hàm lưu lịch sử chat vào file JSON
def save_chat_history(user_id, chat_history):
    file_path = os.path.join(DATA_DIR, f"{user_id}_chat_history.json")
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(chat_history, f, ensure_ascii=False, indent=4)

# Hàm tải lịch sử chat từ file JSON
def load_chat_history(user_id):
    file_path = os.path.join(DATA_DIR, f"{user_id}_chat_history.json")
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

# Hàm xuất toàn bộ lịch sử chat ra file
def export_chat_history(user_id):
    file_path = os.path.join(DATA_DIR, f"{user_id}_chat_history.json")
    export_file = os.path.join(DATA_DIR, f"{user_id}_exported_chat_history.json")
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        with open(export_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        return f"Đã xuất lịch sử chat ra file: {export_file}"
    return "Không có lịch sử chat để xuất."


In [1]:
    # Hàm tạo user_id ngẫu nhiên cho mỗi phiên trò chuyện
    def generate_user_id():
        return str(uuid.uuid4())

    # Khởi tạo hàm trả lời
    def chat(user_input, chat_history, user_id):
        # Nếu user chưa có session thì tạo mới
        if user_id not in user_sessions:
            memory = ConversationBufferMemory()
            chain = ConversationChain(llm=Ollama(model="qwen3:4b"), memory=memory)
            user_sessions[user_id] = chain
            # Tải lịch sử chat từ file nếu có
            chat_history = load_chat_history(user_id)
        else:
            # Lấy lịch sử chat từ state hoặc file
            chat_history = chat_history or load_chat_history(user_id)

        # Lấy chain từ user_sessions
        chain = user_sessions[user_id]
        response = chain.run(user_input)

        # Cập nhật lịch sử chat
        chat_history.append((user_input, response))
        
        # Lưu lịch sử chat vào file
        save_chat_history(user_id, chat_history)
        
        return chat_history, chat_history

    # Hàm xử lý xuất file
    def handle_export(user_id):
        return export_chat_history(user_id)

    # Tạo giao diện
    with gr.Blocks() as demo:
        gr.Markdown("<h2><center>Chatbot Demo</center></h2>")

        chatbot = gr.Chatbot()
        msg = gr.Textbox(placeholder="Nhập câu hỏi...")
        user_id = gr.Textbox(value=generate_user_id(), visible=False)  
        state = gr.State([])  # Lưu lịch sử chat
        send = gr.Button("Gửi")
        export = gr.Button("Xuất lịch sử chat")
        export_output = gr.Textbox(label="Thông báo xuất file", visible=True)

        send.click(
            fn=chat,
            inputs=[msg, state, user_id],
            outputs=[chatbot, state]
        )
        export.click(
            fn=handle_export,
            inputs=[user_id],
            outputs=[export_output]
        )

    demo.launch()

NameError: name 'gr' is not defined

In [5]:
from langchain_community.llms import Ollama

agent_a = Ollama(model="moondream2:latest", temperature=0.7, system="Bạn là AI suy nghĩ logic")
agent_b = Ollama(model="moondream2:latest", temperature=0.7, system="Bạn là AI phản biện")

message = "Hãy giải thích ngắn gọn AI là gì."

max_turns = 3
for turn in range(max_turns):
    try:
        reply_a = agent_a.invoke(message)
        print("Agent A:", reply_a)

        reply_b = agent_b.invoke(reply_a)
        print("Agent B:", reply_b)

        message = reply_b
    except Exception as e:
        print("Lỗi:", e)
        break


Lỗi: Ollama call failed with status code 404. Maybe your model is not found and you should pull the model with `ollama pull moondream2:latest`.


In [4]:
!ollama pull moondream2:latest


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
Error: pull model manifest: file does not exist


In [3]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# =====================
# Load Qwen3:4B model
# =====================
model_name = "llama3.2:latest"  # tên đúng bạn vừa dùng
print(f"Loading model {model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16  # dùng FP16 để nhẹ hơn
)
print("Model loaded successfully!")

# =====================
# Chat function
# =====================
system_message = "You are a helpful assistant"

def message_qwen(prompt, max_length=512):
    """
    Gọi model Qwen3:4B để trả lời văn bản.
    """
    full_prompt = f"{system_message}\nUser: {prompt}\nAssistant:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            top_p=0.9,
            temperature=0.7
        )
        
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("Assistant:")[-1].strip()

# test nhanh
print(message_qwen("Hello, this is a test!"))

# =====================
# Shout function
# =====================
def shout(text):
    print(f"Shout has been called with input: {text}")
    return text.upper()

shout("hello")

# =====================
# Gradio GUI
# =====================
with gr.Blocks() as demo:
    gr.Markdown("## Chat với Qwen3:4B")

    # Tab Chat
    with gr.Tab("Chat Qwen"):
        input_text = gr.Textbox(label="Nhập câu hỏi", lines=2)
        output_text = gr.Textbox(label="Phản hồi")
        input_text.submit(fn=message_qwen, inputs=input_text, outputs=output_text)

    # Tab Shout
    with gr.Tab("Shout Demo"):
        shout_input = gr.Textbox(label="Nhập text")
        shout_output = gr.Textbox(label="Output")
        shout_input.submit(fn=shout, inputs=shout_input, outputs=shout_output)

demo.launch()


Loading model llama3.2:latest ...


OSError: Incorrect path_or_model_id: 'llama3.2:latest'. Please provide either the path to a local folder or the repo_id of a model on the Hub.